In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94
1,IIT: Pct Change due to behavior,1.16,1.14,1.13,1.12,1.11,1.11,1.10,1.10,1.10,1.10,1.12,1.12
2,IIT: Pct Change due to macro,-0.04,-0.05,-0.05,-0.06,-0.07,-0.08,-0.09,-0.10,-0.12,-0.13,-0.08,0.16
3,IIT: Overall Pct Change in taxes,-3.82,-3.85,-3.86,-3.88,-3.90,-3.91,-3.92,-3.94,-3.95,-3.97,-3.90,-3.66
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,0.81,0.85,0.90,0.93,0.95,0.96,0.97,0.97,0.96,0.95,0.93,1.84
6,CIT: Pct Change due to macro,0.63,0.51,0.40,0.32,0.25,0.20,0.16,0.13,0.10,0.08,0.28,-0.87
7,CIT: Overall Pct Change in taxes,1.44,1.36,1.30,1.25,1.20,1.16,1.13,1.09,1.06,1.04,1.20,0.96
8,All: Pct Change due to tax rates,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65
9,All: Pct Change due to behavior,1.14,1.12,1.11,1.11,1.10,1.10,1.10,1.09,1.09,1.09,1.10,1.16


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.23,-0.25,-0.27,-0.28,-0.29,-0.30,-0.31,-0.32,-0.33,-0.35,-2.92
9,Rev Change Due to Behavior,0.06,0.06,0.06,0.07,0.07,0.07,0.07,0.08,0.08,0.08,0.69
10,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.04
11,Total Revenue Change,-0.18,-0.19,-0.21,-0.21,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-2.27


In [5]:
# This csv fild needs to be updated but I ran into an issue with taxcalc

result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)
result_df_static

,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,Total
Base,4.37,4.91,5.11,5.32,5.53,5.76,6.00,6.24,6.49,6.74,56.46
Reform,4.37,4.57,4.77,4.96,5.18,5.40,5.62,5.86,6.10,6.34,53.16
Difference,0.00,-0.34,-0.35,-0.36,-0.35,-0.36,-0.37,-0.38,-0.39,-0.40,-3.31


In [6]:
# This csv fild needs to be updated but I ran into an issue with taxcalc

result_df_dynamic = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_w_behresp.csv', index_col = 0)
result_df_dynamic

,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,Total
Base,4.37,4.91,5.11,5.32,5.53,5.76,6.00,6.24,6.49,6.74,56.46
Reform,4.37,4.64,4.84,5.04,5.25,5.47,5.71,5.94,6.19,6.43,53.88
Difference,0.00,-0.27,-0.27,-0.28,-0.28,-0.28,-0.29,-0.30,-0.30,-0.31,-2.59


In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.22,-0.24,-0.25,-0.26,-0.27,-0.28,-0.30,-0.31,-0.32,-0.33,-2.79
1,Rev Change Due to Behavior,0.05,0.06,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.07,0.63
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.05
3,Total Revenue Change,-0.17,-0.19,-0.20,-0.21,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-2.21


In [8]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = tc_diff * df.loc[1, df_levels.columns[1:]]/df.loc[0, df_levels.columns[1:]]
df_levels.loc[2, df_levels.columns[1:]] = tc_diff * df.loc[2, df_levels.columns[1:]]/df.loc[0, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = tc_diff * (df.loc[0, df_levels.columns[1:]]+df.loc[1, df_levels.columns[1:]]+df.loc[2, df_levels.columns[1:]])/df.loc[0, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.34,-0.35,-0.36,-0.35,-0.36,-0.37,-0.38,-0.39,-0.40,-3.31
1,Rev Change Due to Behavior,-0.00,0.08,0.08,0.08,0.08,0.08,0.08,0.09,0.09,0.09,0.74
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.01,-0.06
3,Total Revenue Change,0.00,-0.26,-0.27,-0.28,-0.28,-0.29,-0.30,-0.31,-0.31,-0.32,-2.62


In [9]:
df_levels.to_csv('og_usa_result_w_tcja.csv')

In [10]:
df_levels['2025-2034'][0]+df_levels['2025-2034'][1]

-2.620928417595902